# Exercice 3 — Solution

In [ ]:
#r "nuget: Microsoft.SemanticKernel, 1.74.0"
#r "nuget: Microsoft.SemanticKernel.Connectors.Ollama, 1.74.0-alpha"
#r "nuget: Microsoft.SemanticKernel.Connectors.Qdrant, 1.74.0-preview"
#pragma warning disable SKEXP0070,SKEXP0001
using Microsoft.Extensions.AI; using Microsoft.Extensions.VectorData; using Microsoft.SemanticKernel; using Microsoft.SemanticKernel.Connectors.Qdrant; using Qdrant.Client;
using System.Security.Cryptography; using System.Text;
var endpoint=new Uri("http://localhost:11434");
var embed=Kernel.CreateBuilder().AddOllamaEmbeddingGenerator("nomic-embed-text",endpoint).Build().GetRequiredService<IEmbeddingGenerator<string,Embedding<float>>>();
var store=new QdrantVectorStore(new QdrantClient("localhost"),ownsClient:true);
ulong Id(string s)=>BitConverter.ToUInt64(MD5.HashData(Encoding.UTF8.GetBytes(s)),0);

In [ ]:
public sealed class Doc{ [VectorStoreKey] public ulong Id{get;set;} [VectorStoreData] public string Text{get;set;}="";
  [VectorStoreVector(768,DistanceFunction=DistanceFunction.CosineSimilarity)] public ReadOnlyMemory<float> Vector{get;set;} }

In [ ]:
var col=store.GetCollection<ulong,Doc>("ex3"); await col.EnsureCollectionExistsAsync();
async Task Add(string t){ var v=(await embed.GenerateAsync(t)).Vector; await col.UpsertAsync(new Doc{Id=Id(t),Text=t,Vector=v}); }
await Add("Le cafe est une boisson chaude a base de grains torrefies.");
await Add("Le the est une infusion chaude de feuilles.");
await Add("Le velo est un moyen de transport a deux roues.");
var qv=(await embed.GenerateAsync("boisson chaude")).Vector;
await foreach(var r in col.SearchAsync(qv,top:3)) Console.WriteLine($"[{r.Score:0.000}] {r.Record.Text}");